# Antimeridian bbox test: pystac → stac-geoparquet → rustac search

Creates a `pystac.Item` with an antimeridian-crossing bbox, writes it to a local
stac-geoparquet file, then uses `rustac` to run point-intersection searches.

**Note on the bbox:** a literal `[170, -10, 170, 10]` has `west == east` (a
zero-width box), which can't be the intended antimeridian-crossing shape. This
notebook uses the reversed-longitude convention from RFC 7946 §5.2 / the STAC
API spec instead: **`[170, -10, -170, 10]`** (west=170°, east=-170°), which
covers the ~20°-wide sliver straddling ±180° between 170°E and 170°W —
consistent with the Fiji-straddling example used throughout this conversation.

Expected results for the three test points:
- `[175, -5]` → **True** (170° to 180° side of the crossing)
- `[-175, 5]` → **True** (-180° to -170° side of the crossing)
- `[0, 0]` → **False** (nowhere near the antimeridian)

In [ ]:
# Install dependencies (skip if already installed)
%pip install -q pystac rustac

In [1]:
from datetime import datetime, timezone

import pystac
import rustac

## 1. Build the pystac Item

`bbox` uses the reversed convention (`west=170 > east=-170`). `geometry` is a
`MultiPolygon` split at the seam — one piece from 170° to 180°, one piece from
-180° to -170° — which is the standard way to represent an antimeridian-crossing
footprint without a self-intersecting ring. pystac does not compute or validate
this for you (see earlier discussion): both `bbox` and `geometry` are supplied
explicitly here.

In [2]:
bbox = [170, -10, -170, 10]  # west, south, east, north (reversed: west > east)

geometry = {
    "type": "MultiPolygon",
    "coordinates": [
        # West piece: 170°E to 180°
        [[[170, -10], [180, -10], [180, 10], [170, 10], [170, -10]]],
        # East piece: -180° to -170° (i.e. 170°W)
        [[[-180, -10], [-170, -10], [-170, 10], [-180, 10], [-180, -10]]],
    ],
}

item = pystac.Item(
    id="antimeridian-test-item",
    geometry=geometry,
    bbox=bbox,
    datetime=datetime.now(timezone.utc),
    properties={},
)

item_dict = item.to_dict()
print("Item id:  ", item_dict["id"])
print("Item bbox:", item_dict["bbox"])

Item id:   antimeridian-test-item
Item bbox: [170, -10, -170, 10]


In [3]:
item_dict

{'type': 'Feature',
 'stac_version': '1.1.0',
 'stac_extensions': [],
 'id': 'antimeridian-test-item',
 'geometry': {'type': 'MultiPolygon',
  'coordinates': [[[[170, -10], [180, -10], [180, 10], [170, 10], [170, -10]]],
   [[[-180, -10], [-170, -10], [-170, 10], [-180, 10], [-180, -10]]]]},
 'bbox': [170, -10, -170, 10],
 'properties': {'datetime': '2026-09-21T04:25:53.629291Z'},
 'links': [],
 'assets': {}}

## 2. Write to a local stac-geoparquet file

`rustac.write` takes a list of STAC item dicts and writes them to a
stac-geoparquet file. `rustac`'s async functions can be awaited directly at
the top level of a notebook cell.

In [4]:
parquet_path = "items.parquet"

await rustac.write(parquet_path, [item_dict])
print(f"Wrote 1 item to {parquet_path}")

Wrote 1 item to items.parquet


### Sanity check: read it back

Confirms the reversed bbox and split `MultiPolygon` geometry survive the
geoparquet round-trip unchanged.

In [5]:
read_back = await rustac.read(parquet_path)
feature = read_back["features"][0]

print("bbox from geoparquet:    ", feature["bbox"])
print("geometry from geoparquet:", feature["geometry"])

bbox from geoparquet:     (170.0, -10.0, -170.0, 10.0)
geometry from geoparquet: {'type': 'MultiPolygon', 'coordinates': [[[[170.0, -10.0], [180.0, -10.0], [180.0, 10.0], [170.0, 10.0], [170.0, -10.0]]], [[[-180.0, -10.0], [-170.0, -10.0], [-170.0, 10.0], [-180.0, 10.0], [-180.0, -10.0]]]]}


## 3. Search with rustac for each test point

`rustac.search` can query a local stac-geoparquet file directly (no STAC API
server needed) using `intersects` with a GeoJSON `Point`. Under the hood this
uses DuckDB's spatial extension, which is downloaded automatically on first
use — that first call needs internet access.

In [15]:
test_points = {
    "[175, -5]  (east hemisphere piece, 170→180)": ([175, -5], 1),
    "[-175, 5]  (west hemisphere piece, -180→-170)": ([-175, 5], 1),
    "[165, 5]   (not intersecting)": ([165, 5], 0),
    "[180, 0]   (interesection on AM)": ([180, 0], 1),
}

results = {}

for label, (coords, expected) in test_points.items():
    result = await rustac.search(
        parquet_path,
        intersects={"type": "Point", "coordinates": coords},
    )
    matched = len(result)
    results[label] = matched
    status = "✅" if matched == expected else "❌"
    print(f"{status} {label:38s} matched={matched}  expected={expected}")

✅ [175, -5]  (east hemisphere piece, 170→180) matched=1  expected=1
✅ [-175, 5]  (west hemisphere piece, -180→-170) matched=1  expected=1
✅ [165, 5]   (not intersecting)          matched=0  expected=0
✅ [180, 0]   (interesection on AM)       matched=1  expected=1


## 4. Assert the expected pattern: True, True, False

In [ ]:
expected_pattern = [True, True, False]
actual_pattern = list(results.values())

print("Expected:", expected_pattern)
print("Actual:  ", actual_pattern)

assert actual_pattern == expected_pattern, (
    "rustac's search over the antimeridian-crossing item did not match the "
    "expected True/True/False pattern — the reversed bbox/geometry may not be "
    "handled correctly."
)
print("\nAll assertions passed — rustac correctly resolves the antimeridian-crossing item.")